In [1]:
import pandas as pd
import time

start_time = time.time()

df_1000 = pd.read_csv(r"D:\personal-project\food-online-test\data\customers-100000.csv")


end_duration = time.time() - start_time
print("Processing End Duration: ", round(end_duration, 2), "seconds")

Processing End Duration:  1.02 seconds


In [2]:
df_1000.head(5)

,Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
0,1,ffeCAb7AbcB0f07,Jared,Jarvis,Sanchez-Fletcher,Hatfieldshire,Eritrea,274.188.8773x41185,001-215-760-4642x969,gabriellehartman@benjamin.com,2021-11-11,https://www.mccarthy.info/
1,2,b687FfC4F1600eC,Marie,Malone,Mckay PLC,Robertsonburgh,Botswana,283-236-9529,(189)129-8356x63741,kstafford@sexton.com,2021-05-14,http://www.reynolds.com/
2,3,9FF9ACbc69dcF9c,Elijah,Barrera,Marks and Sons,Kimbury,Barbados,8252703789,459-916-7241x0909,jeanettecross@brown.com,2021-03-17,https://neal.com/
3,4,b49edDB1295FF6E,Sheryl,Montgomery,"Kirby, Vaughn and Sanders",Briannaview,Antarctica (the territory South of 60 deg S),425.475.3586,(392)819-9063,thomassierra@barrett.com,2020-09-23,https://www.powell-bryan.com/
4,5,3dcCbFEB17CCf2E,Jeremy,Houston,Lester-Manning,South Brianna,Micronesia,+1-223-666-5313x4530,252-488-3850x692,rubenwatkins@jacobs-wallace.info,2020-09-18,https://www.carrillo.com/


Based on the 'Head' data above, we can see that 'Phone 1' and 'Phone 2' has untidy data, sometimes it has 'x', '.', and even has '-'. We know that phone number usually only has '-' and '+', so then I'll change 'x' and '.' into '-'

In [3]:
import re

cols = ["Phone 1", "Phone 2"]

# Function to know the pattern of each phone number
def get_phone_pattern(phone):
    if pd.isna(phone):
        return "MISSING"

    phone = str(phone).strip()

    # Change every digit with '9'
    return re.sub(r"\d", "9", phone)

for col in cols:
    print(f"\nPattern")
    print(
        df_1000[col]
        .apply(get_phone_pattern)
        .value_counts()
        .head(50)
    )


Pattern
Phone 1
(999)999-9999             8136
999.999.9999              8121
999-999-9999              7947
9999999999                7893
999-999-9999x9999         4106
+9-999-999-9999x99999     4072
999.999.9999x9999         4051
(999)999-9999x9999        4049
+9-999-999-9999x9999      4041
999-999-999-9999x999      4039
(999)999-9999x999         4019
(999)999-9999x99999       3998
999-999-9999x999          3991
999.999.9999x99999        3983
+9-999-999-9999           3982
999-999-9999x99999        3972
999-999-999-9999          3967
999-999-999-9999x9999     3951
999-999-999-9999x99999    3946
999.999.9999x999          3880
+9-999-999-9999x999       3856
Name: count, dtype: int64

Pattern
Phone 2
999-999-9999              8108
(999)999-9999             8034
9999999999                8011
999.999.9999              7807
(999)999-9999x99999       4114
999-999-9999x999          4069
+9-999-999-9999x999       4068
999-999-999-9999x9999     4056
999-999-9999x99999        4054
999-999-99

In [4]:
df_1000[cols] = df_1000[cols].replace(
    r"[.xX]", "-", regex=True
)

for col in cols:
    print("Pattern")
    print(
        df_1000[col]
        .apply(get_phone_pattern)
        .value_counts()
        .head(50)
    )

Pattern
Phone 1
999-999-9999              16068
999-999-9999-9999          8157
(999)999-9999              8136
999-999-9999-99999         7955
9999999999                 7893
999-999-9999-999           7871
+9-999-999-9999-99999      4072
(999)999-9999-9999         4049
+9-999-999-9999-9999       4041
999-999-999-9999-999       4039
(999)999-9999-999          4019
(999)999-9999-99999        3998
+9-999-999-9999            3982
999-999-999-9999           3967
999-999-999-9999-9999      3951
999-999-999-9999-99999     3946
+9-999-999-9999-999        3856
Name: count, dtype: int64
Pattern
Phone 2
999-999-9999              15915
999-999-9999-999           8041
999-999-9999-99999         8037
(999)999-9999              8034
9999999999                 8011
999-999-9999-9999          7952
(999)999-9999-99999        4114
+9-999-999-9999-999        4068
999-999-999-9999-9999      4056
999-999-999-9999           4002
999-999-999-9999-99999     4000
(999)999-9999-999          3995
+9-999-999-999

After we change '.' and 'x' into '-', we can see that the pattern variety is tinier than before.

In [5]:
df_phone = df_1000[["Country", "Phone 1"]].copy()

df_phone["Phone 1"] = df_phone["Phone 1"].astype("string").str.strip()

# Ambil nomor utama sebelum ekstensi
df_phone["main_number"] = df_phone["Phone 1"].str.replace(
    r"[xX]\d+$", "", regex=True
)

# Ambil kode telepon internasional jika diawali +
df_phone["international_prefix"] = df_phone["main_number"].str.extract(
    r"^\s*(\+\d+)"
)

display(
    df_phone.groupby(["Country", "international_prefix"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["Country", "count"], ascending=[True, False])
)

,Country,international_prefix,count
1,Afghanistan,<NA>,348
0,Afghanistan,+1,69
3,Albania,<NA>,347
2,Albania,+1,74
5,Algeria,<NA>,345
...,...,...,...
480,Yemen,+1,62
483,Zambia,<NA>,348
482,Zambia,+1,70
485,Zimbabwe,<NA>,320


In [6]:
df_phone["main_number"].str.extract(
    r"^\s*(\+\d+)"
)[0].value_counts(dropna=False)

0
<NA>    84049
+1      15951
Name: count, dtype: Int64

In [7]:
prefix_by_country = pd.crosstab(
    df_phone["Country"],
    df_phone["main_number"].str.startswith("+1", na=False),
    normalize="index"
).mul(100).round(2)

prefix_by_country.columns = ["Tanpa +1", "Dengan +1"]

display(prefix_by_country)

,Tanpa +1,Dengan +1
Country,,
Afghanistan,83.45,16.55
Albania,82.42,17.58
Algeria,79.31,20.69
American Samoa,86.23,13.77
Andorra,84.75,15.25
...,...,...
Wallis and Futuna,84.41,15.59
Western Sahara,84.29,15.71
Yemen,84.65,15.35


In [8]:
df_1000["Company"].value_counts()

Company
Campbell Ltd                  17
Wilkerson Ltd                 17
Booker and Sons               16
Acosta Ltd                    16
Mccarty and Sons              15
                              ..
Delgado-Daniels                1
Wright, Erickson and Wells     1
Bryant-Jordan                  1
Rubio-Sutton                   1
Henry, Murray and Gomez        1
Name: count, Length: 71994, dtype: int64

In [9]:
df_1000["City"].value_counts()

City
Lake Frederick      16
East Jeremy         15
East Lee            15
West Alec           15
New Christopher     15
                    ..
South Lauraland      1
Mackland             1
Port Anaburgh        1
Warrenstad           1
East Rosechester     1
Name: count, Length: 49154, dtype: int64

In [10]:
df_1000["Country"].value_counts()

Country
Congo               835
Korea               820
Saudi Arabia        463
Pitcairn Islands    456
Saint Martin        453
                   ... 
Moldova             371
Jordan              365
Saint Lucia         365
Slovenia            361
Greece              359
Name: count, Length: 243, dtype: int64

In [11]:
summary = pd.DataFrame({
    "unique_values": df_1000.nunique(),
    "unique_ratio_pct": (
        df_1000.nunique() / len(df_1000) * 100
    ).round(2)
})

display(summary.sort_values("unique_ratio_pct", ascending=False))

,unique_values,unique_ratio_pct
Index,100000,100.00
Customer Id,100000,100.00
Phone 2,100000,100.00
Phone 1,100000,100.00
Email,99995,100.00
Company,71994,71.99
Website,50471,50.47
City,49154,49.15
Last Name,1000,1.00
Subscription Date,880,0.88


In [14]:
df_1000['Subscription Date'] = pd.to_datetime(df_1000['Subscription Date'], errors='coerce')

d = df_1000['Subscription Date'].dt
df_1000['subscription_year']    = d.year
df_1000['subscription_month']   = d.month
df_1000['month_name']           = d.month_name()        # "January", ... (aman untuk NaT)
df_1000['subscription_weekday'] = d.day_name()          # "Monday", ...
df_1000['year_month']           = d.to_period('M')      # 2021-07 -> untuk tren bulanan

In [17]:
def analyze_trends(df, group_cols):
    df = df.copy()
    df['Subscription Date'] = pd.to_datetime(df['Subscription Date'], errors='coerce')
    df['Period'] = df['Subscription Date'].dt.to_period('M')

    # Validate columns
    missing_cols = [col for col in group_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Column(s) not found: {', '.join(missing_cols)}")

    # Drop missing dates/hours
    df = df.dropna(subset=['Subscription Date'])

    # Total count per group
    grouped = df.groupby(group_cols).size().reset_index(name='Count')
    total = grouped['Count'].sum()
    grouped['Percent'] = (grouped['Count'] / total * 100).round(2)

    # Monthly trend
    monthly_trend = (
        df
        .groupby([df['Period'].dt.strftime('%m-%Y')] + group_cols)
        .size()
        .reset_index(name='Count')
        .pivot_table(index=group_cols, columns='Period', values='Count', fill_value=0)
        .reset_index()
    )

    return grouped, monthly_trend

group_columns = ['First Name', 'Country']
summaries = {}
monthly_trends = {}
hourly_trends = {}

for col in group_columns:
    summary, monthly_trend = analyze_trends(
        df_1000,
        group_cols=[col]  # Only one column at a time
    )

    summaries[col] = summary
    monthly_trends[col] = monthly_trend

In [18]:
# Show summary for Classification
print("=== Classification Summary ===")
display(summaries['First Name'])

# Show trend for Classification
print("=== Classification Monthly Trend ===")
display(monthly_trends['First Name'])


# Show summary for View
print("=== View Summary ===")
display(summaries['Country'])

# Show trend for View
print("=== View Monthly Trend ===")
display(monthly_trends['Country'])

=== Classification Summary ===


,First Name,Count,Percent
0,Aaron,147,0.15
1,Abigail,139,0.14
2,Adam,138,0.14
3,Adrian,160,0.16
4,Adriana,155,0.16
...,...,...,...
685,Yolanda,147,0.15
686,Yvette,131,0.13
687,Yvonne,144,0.14
688,Zachary,127,0.13


=== Classification Monthly Trend ===


Period,First Name,01-2020,01-2021,01-2022,02-2020,02-2021,02-2022,03-2020,03-2021,03-2022,...,08-2020,08-2021,09-2020,09-2021,10-2020,10-2021,11-2020,11-2021,12-2020,12-2021
0,Aaron,3.0,3.0,5.0,7.0,4.0,6.0,5.0,3.0,4.0,...,4.0,2.0,6.0,5.0,10.0,4.0,5.0,6.0,3.0,6.0
1,Abigail,8.0,5.0,4.0,6.0,6.0,3.0,13.0,6.0,3.0,...,6.0,4.0,1.0,3.0,4.0,6.0,4.0,4.0,6.0,9.0
2,Adam,10.0,5.0,5.0,5.0,3.0,6.0,2.0,8.0,7.0,...,1.0,2.0,6.0,2.0,2.0,5.0,5.0,4.0,5.0,10.0
3,Adrian,9.0,6.0,5.0,4.0,6.0,6.0,2.0,4.0,4.0,...,5.0,5.0,2.0,5.0,12.0,9.0,5.0,10.0,6.0,10.0
4,Adriana,4.0,6.0,4.0,2.0,2.0,5.0,6.0,8.0,4.0,...,9.0,2.0,7.0,5.0,8.0,11.0,7.0,7.0,2.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,Yolanda,2.0,4.0,2.0,4.0,6.0,4.0,5.0,3.0,5.0,...,4.0,4.0,4.0,3.0,5.0,7.0,5.0,11.0,5.0,8.0
686,Yvette,6.0,4.0,9.0,7.0,4.0,2.0,4.0,3.0,2.0,...,7.0,4.0,4.0,5.0,6.0,5.0,2.0,4.0,7.0,4.0
687,Yvonne,1.0,5.0,5.0,3.0,4.0,7.0,9.0,5.0,4.0,...,7.0,3.0,6.0,6.0,7.0,4.0,4.0,5.0,4.0,5.0
688,Zachary,4.0,8.0,1.0,6.0,5.0,6.0,1.0,1.0,4.0,...,3.0,5.0,6.0,1.0,3.0,7.0,8.0,4.0,9.0,5.0


=== View Summary ===


,Country,Count,Percent
0,Afghanistan,417,0.42
1,Albania,421,0.42
2,Algeria,435,0.43
3,American Samoa,443,0.44
4,Andorra,413,0.41
...,...,...,...
238,Wallis and Futuna,404,0.40
239,Western Sahara,420,0.42
240,Yemen,404,0.40
241,Zambia,418,0.42


=== View Monthly Trend ===


Period,Country,01-2020,01-2021,01-2022,02-2020,02-2021,02-2022,03-2020,03-2021,03-2022,...,08-2020,08-2021,09-2020,09-2021,10-2020,10-2021,11-2020,11-2021,12-2020,12-2021
0,Afghanistan,16.0,12.0,20.0,13.0,11.0,14.0,12.0,10.0,17.0,...,11.0,11.0,19.0,16.0,15.0,25.0,9.0,19.0,16.0,18.0
1,Albania,14.0,23.0,16.0,8.0,18.0,16.0,13.0,11.0,14.0,...,14.0,16.0,10.0,17.0,11.0,15.0,10.0,8.0,19.0,8.0
2,Algeria,9.0,12.0,17.0,14.0,17.0,14.0,10.0,20.0,21.0,...,20.0,17.0,19.0,12.0,21.0,13.0,11.0,19.0,13.0,15.0
3,American Samoa,16.0,14.0,7.0,11.0,9.0,16.0,11.0,17.0,20.0,...,14.0,8.0,14.0,18.0,12.0,21.0,17.0,19.0,18.0,18.0
4,Andorra,13.0,12.0,21.0,14.0,13.0,17.0,18.0,13.0,16.0,...,15.0,15.0,16.0,12.0,12.0,11.0,19.0,10.0,10.0,19.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,Wallis and Futuna,16.0,10.0,13.0,12.0,10.0,18.0,11.0,12.0,20.0,...,11.0,18.0,15.0,18.0,19.0,14.0,25.0,14.0,14.0,13.0
239,Western Sahara,23.0,16.0,15.0,13.0,10.0,17.0,17.0,16.0,14.0,...,12.0,14.0,12.0,14.0,12.0,9.0,18.0,10.0,14.0,17.0
240,Yemen,19.0,18.0,16.0,11.0,12.0,17.0,11.0,11.0,15.0,...,13.0,16.0,17.0,20.0,11.0,14.0,20.0,14.0,8.0,11.0
241,Zambia,17.0,13.0,16.0,18.0,13.0,10.0,18.0,16.0,13.0,...,19.0,9.0,8.0,16.0,15.0,21.0,20.0,11.0,13.0,9.0
